# Processing Notebook

Notebook nay tu chua toan bo logic preprocessing.
Ban co the chay preprocessing tu raw CSV ngay trong notebook ma khong can phu thuoc vao module Python rieng.


In [15]:
from pathlib import Path
from datetime import datetime
import hashlib
import json
import logging
import shutil
import sys
import warnings

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("preprocess_notebook")


def resolve_repo_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent, cwd.parent.parent]
    for candidate in candidates:
        if (candidate / "data").exists() and (candidate / "code").exists():
            return candidate
    return cwd


REPO_ROOT = resolve_repo_root()
print("Repo root :", REPO_ROOT)


Repo root : C:\Users\Admin\Documents\GitHub\Airline_OTP_Analysis


In [16]:
# Preprocessing constants
FILES = [
    "data/raw/T_ONTIME_REPORTING_2021.csv",
    "data/raw/T_ONTIME_REPORTING_2022.csv",
    "data/raw/T_ONTIME_REPORTING_2023.csv",
    "data/raw/T_ONTIME_REPORTING_2024.csv",
    "data/raw/T_ONTIME_REPORTING_2025.csv",
]

CHUNKSIZE = 700_000
TARGET = "ARR_DEL15"
TRAIN_YEARS = {2021, 2022, 2023, 2024}
TEST_YEARS = {2025}

CANCEL_NULL_COLS = [
    "DEP_TIME", "ARR_TIME", "WHEELS_OFF", "WHEELS_ON", "TAXI_OUT", "TAXI_IN",
    "ACTUAL_ELAPSED_TIME", "AIR_TIME",
    "DEP_DELAY", "DEP_DELAY_NEW", "DEP_DEL15", "DEP_DELAY_GROUP",
    "ARR_DELAY", "ARR_DELAY_NEW", "ARR_DEL15", "ARR_DELAY_GROUP",
    "CARRIER_DELAY", "WEATHER_DELAY", "NAS_DELAY", "SECURITY_DELAY", "LATE_AIRCRAFT_DELAY",
    "FIRST_DEP_TIME", "TOTAL_ADD_GTIME", "LONGEST_ADD_GTIME",
]
DIVERT_NULL_COLS = [
    "ARR_TIME", "ARR_DELAY", "ARR_DELAY_NEW", "ARR_DEL15", "ARR_DELAY_GROUP", "ARR_TIME_BLK",
    "WHEELS_ON", "TAXI_IN",
]
HHMM_COLS = [
    "CRS_DEP_TIME", "CRS_ARR_TIME",
]
COLUMNS_TO_DROP = [
    "OP_CARRIER_AIRLINE_ID", "TAIL_NUM", "FLIGHTS", "OP_CARRIER_FL_NUM",
    "ORIGIN_AIRPORT_ID", "ORIGIN_AIRPORT_SEQ_ID", "ORIGIN_CITY_MARKET_ID",
    "ORIGIN_STATE_FIPS", "ORIGIN_WAC",
    "DEST_AIRPORT_ID", "DEST_AIRPORT_SEQ_ID", "DEST_CITY_MARKET_ID",
    "DEST_STATE_FIPS", "DEST_WAC",
    "ACTUAL_ELAPSED_TIME", "AIR_TIME", "ARR_DELAY_GROUP", "DEP_DELAY_GROUP", "ARR_TIME_BLK",
]
DELAY_CAUSE_COLS = [
    "CARRIER_DELAY", "WEATHER_DELAY", "NAS_DELAY", "SECURITY_DELAY", "LATE_AIRCRAFT_DELAY",
]
FREQ_ENCODE_COLS = ["OP_CARRIER", "ORIGIN", "DEST", "ROUTE", "DEP_TIME_BLK"]
OTP_GROUP_COLS = ["ORIGIN", "OP_CARRIER"]
# DUPLICATE_KEY_COLS removed

DTYPE_MAP_INT = {
    "YEAR": "Int16", "DAY_OF_MONTH": "Int16", "DAY_OF_WEEK": "Int16",
    "OP_CARRIER_AIRLINE_ID": "Int32", "OP_CARRIER_FL_NUM": "Int32",
    "ORIGIN_AIRPORT_ID": "Int32", "ORIGIN_AIRPORT_SEQ_ID": "Int32",
    "ORIGIN_CITY_MARKET_ID": "Int32", "ORIGIN_STATE_FIPS": "Int16", "ORIGIN_WAC": "Int16",
    "DEST_AIRPORT_ID": "Int32", "DEST_AIRPORT_SEQ_ID": "Int32",
    "DEST_CITY_MARKET_ID": "Int32", "DEST_STATE_FIPS": "Int16", "DEST_WAC": "Int16",
    "DEP_DEL15": "Int16", "DEP_DELAY_GROUP": "Int16",
    "ARR_DEL15": "Int16", "ARR_DELAY_GROUP": "Int16",
    "CANCELLED": "Int16", "DIVERTED": "Int16",
    "DISTANCE_GROUP": "Int16", "FLIGHTS": "Int16",
    "DIV_AIRPORT_LANDINGS": "Int16", "DIV_REACHED_DEST": "Int16",
}
DTYPE_MAP_FLOAT = {
    "DEP_DELAY": "float32", "DEP_DELAY_NEW": "float32",
    "ARR_DELAY": "float32", "ARR_DELAY_NEW": "float32",
    "TAXI_OUT": "float32", "TAXI_IN": "float32",
    "CRS_ELAPSED_TIME": "float32", "ACTUAL_ELAPSED_TIME": "float32",
    "AIR_TIME": "float32", "DISTANCE": "float32",
    "CARRIER_DELAY": "float32", "WEATHER_DELAY": "float32",
    "NAS_DELAY": "float32", "SECURITY_DELAY": "float32", "LATE_AIRCRAFT_DELAY": "float32",
    "TOTAL_ADD_GTIME": "float32", "LONGEST_ADD_GTIME": "float32",
    "DIV_ACTUAL_ELAPSED_TIME": "float32", "DIV_ARR_DELAY": "float32", "DIV_DISTANCE": "float32",
}
STR_COLS = {
    "OP_UNIQUE_CARRIER", "OP_CARRIER", "TAIL_NUM", "ORIGIN", "ORIGIN_CITY_NAME",
    "ORIGIN_STATE_ABR", "ORIGIN_STATE_NM", "DEST", "DEST_CITY_NAME",
    "DEST_STATE_ABR", "DEST_STATE_NM", "DEP_TIME_BLK", "ARR_TIME_BLK", "CANCELLATION_CODE",
}

TRACK_A_FEATURES = [
    "YEAR", "DAY_OF_MONTH", "DAY_OF_WEEK", "IS_WEEKEND",
    "CRS_DEP_TIME_MIN", "CRS_ARR_TIME_MIN",
    "CRS_DEP_SIN", "CRS_DEP_COS", "CRS_ARR_SIN", "CRS_ARR_COS",
    "DISTANCE",
    "OP_CARRIER_FREQ", "CARRIER_HIST_OTP",
    "ORIGIN_FREQ", "ORIGIN_HIST_OTP", "DEST_FREQ",
    "ROUTE_FREQ", "DEP_TIME_BLK_FREQ",
]
TRACK_B_EXTRA = ["DEP_DELAY", "TAXI_OUT"]
TRACK_B_ENGINEERED = ["IS_HEAVY_DELAY"]
TRACK_A_FORBIDDEN = {"DEP_TIME", "DEP_DELAY", "DEP_DELAY_NEW", "DEP_DEL15", "DEP_DELAY_GROUP", "TAXI_OUT", "WHEELS_OFF", "WHEELS_ON", "TAXI_IN", "ARR_TIME", "ARR_DELAY", "ARR_DELAY_NEW", "ARR_DEL15", "ARR_DELAY_GROUP", "ACTUAL_ELAPSED_TIME", "AIR_TIME", "CARRIER_DELAY", "WEATHER_DELAY", "NAS_DELAY", "SECURITY_DELAY", "LATE_AIRCRAFT_DELAY", "FIRST_DEP_TIME", "TOTAL_ADD_GTIME", "LONGEST_ADD_GTIME", "CANCELLED", "DIVERTED"}
TRACK_B_FORBIDDEN = {"ARR_TIME", "ARR_DELAY", "ARR_DELAY_NEW", "ARR_DEL15", "ARR_DELAY_GROUP", "ARR_TIME_BLK", "ACTUAL_ELAPSED_TIME", "AIR_TIME", "CARRIER_DELAY", "WEATHER_DELAY", "NAS_DELAY", "SECURITY_DELAY", "LATE_AIRCRAFT_DELAY", "WHEELS_ON", "TAXI_IN", "FIRST_DEP_TIME", "TOTAL_ADD_GTIME", "LONGEST_ADD_GTIME", "CANCELLED", "DIVERTED"}
DERIVED_COLUMNS = {"FL_DATE", "IS_WEEKEND", "ROUTE", "CRS_DEP_TIME_MIN", "CRS_ARR_TIME_MIN", "CRS_DEP_SIN", "CRS_DEP_COS", "CRS_ARR_SIN", "CRS_ARR_COS"}


In [17]:
# Utility functions copied from utils.py, transformations.py, and ml_preparation.py

def hash_inputs(paths):
    h = hashlib.md5()
    for p in sorted(paths):
        h.update(str(p).encode())
    return h.hexdigest()


def hhmm_to_minutes(s):
    v = pd.to_numeric(s, errors="coerce")
    hh = (v // 100).astype("Int32")
    mm = (v % 100).astype("Int32")
    result = hh * 60 + mm
    result = result.where((result >= 0) & (result <= 1439), other=pd.NA)
    return result


def cyclic_encode(minutes_col, period=1440):
    rad = 2 * np.pi * minutes_col / period
    return np.sin(rad).astype("float32"), np.cos(rad).astype("float32")


def standardize_columns(df):
    df.columns = df.columns.str.strip().str.upper()
    df = df[[c for c in df.columns if not c.startswith("UNNAMED")]]
    return df


def cast_dtypes(df):
    for c, dt in DTYPE_MAP_INT.items():
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").astype(dt)
    for c, dt in DTYPE_MAP_FLOAT.items():
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").astype(dt)
    for c in STR_COLS:
        if c in df.columns:
            df[c] = df[c].astype("string")
    return df


def apply_r0_duplicates(df, rule_counts=None):
    full_removed = int(df.duplicated().sum())
    if full_removed:
        df = df.drop_duplicates().copy()
    if rule_counts is not None:
        rule_counts["R0"] = rule_counts.get("R0", 0) + full_removed
    return df


def apply_r1_cancelled(df):
    mask = df["CANCELLED"] == 1
    n = int(mask.sum())
    div_cols = [c for c in df.columns if c.startswith("DIV_")]
    for c in CANCEL_NULL_COLS + div_cols:
        if c in df.columns:
            df.loc[mask, c] = pd.NA if df[c].dtype.name.startswith(("Int", "string")) else np.nan
    return df, n


def apply_r2_diverted(df):
    mask = df["DIVERTED"] == 1
    n = int(mask.sum())
    for c in DIVERT_NULL_COLS:
        if c in df.columns:
            df.loc[mask, c] = pd.NA if df[c].dtype.name.startswith(("Int", "string")) else np.nan
    return df, n


def apply_r3_hhmm(df, stats):
    for c in HHMM_COLS:
        if c not in df.columns:
            continue
        mc = c + "_MIN"
        df[mc] = hhmm_to_minutes(df[c])
        parsed = int(df[mc].notna().sum())
        na = int(df[mc].isna().sum())
        stats.setdefault(c, {"parsed": 0, "na": 0})
        stats[c]["parsed"] += parsed
        stats[c]["na"] += na
        df[mc] = df[mc].astype("Int32")
    if "CRS_DEP_TIME_MIN" in df.columns:
        s, co = cyclic_encode(df["CRS_DEP_TIME_MIN"].astype("float32"))
        df["CRS_DEP_SIN"] = s
        df["CRS_DEP_COS"] = co
    if "CRS_ARR_TIME_MIN" in df.columns:
        s, co = cyclic_encode(df["CRS_ARR_TIME_MIN"].astype("float32"))
        df["CRS_ARR_SIN"] = s
        df["CRS_ARR_COS"] = co
    return df


def apply_r4_date(df, dow_mismatches):
    df["FL_DATE"] = pd.to_datetime(df["YEAR"].astype(str) + "-01-" + df["DAY_OF_MONTH"].astype(str), errors="coerce")
    computed_dow = df["FL_DATE"].dt.dayofweek.astype("Int16")
    bts_dow = df["DAY_OF_WEEK"] - 1
    mismatch = (bts_dow != computed_dow) & bts_dow.notna() & computed_dow.notna()
    dow_mismatches["count"] = dow_mismatches.get("count", 0) + int(mismatch.sum())
    df["DAY_OF_WEEK"] = computed_dow
    return df


def apply_r5_weekend(df):
    df["IS_WEEKEND"] = (df["DAY_OF_WEEK"].isin({5, 6})).astype("Int16")
    return df


def apply_r6_route(df):
    df["ROUTE"] = df["ORIGIN"].astype(str) + "-" + df["DEST"].astype(str)
    return df



def clean_chunk(df, target_col, hhmm_stats, dow_mismatches, rule_counts):
    df = apply_r0_duplicates(df, rule_counts)

    df, n1 = apply_r1_cancelled(df)
    rule_counts["R1"] = rule_counts.get("R1", 0) + n1
    df, n2 = apply_r2_diverted(df)
    rule_counts["R2"] = rule_counts.get("R2", 0) + n2
    apply_r3_hhmm(df, hhmm_stats)
    rule_counts["R3"] = rule_counts.get("R3", 0) + len(df)
    apply_r4_date(df, dow_mismatches)
    rule_counts["R4"] = rule_counts.get("R4", 0) + len(df)
    apply_r5_weekend(df)
    apply_r6_route(df)
    return df


def apply_freq_otp(df, freq_maps, otp_maps, global_otp, unseen_log):
    col_map = {"OP_CARRIER": "OP_CARRIER_FREQ", "ORIGIN": "ORIGIN_FREQ", "DEST": "DEST_FREQ", "ROUTE": "ROUTE_FREQ", "DEP_TIME_BLK": "DEP_TIME_BLK_FREQ"}
    for src, dst in col_map.items():
        if src in df.columns and src in freq_maps:
            mapped = df[src].map(freq_maps[src])
            unseen = mapped.isna() & df[src].notna()
            unseen_log.setdefault(src, {"total": 0, "unseen": 0})
            unseen_log[src]["total"] += int(df[src].notna().sum())
            unseen_log[src]["unseen"] += int(unseen.sum())
            df[dst] = mapped.fillna(0).astype("float32")
    otp_col_map = {"ORIGIN": "ORIGIN_HIST_OTP", "OP_CARRIER": "CARRIER_HIST_OTP"}
    for src, dst in otp_col_map.items():
        if src in df.columns and src in otp_maps:
            mapped = df[src].map(otp_maps[src])
            unseen = mapped.isna() & df[src].notna()
            unseen_log.setdefault(f"otp_{src}", {"total": 0, "unseen": 0})
            unseen_log[f"otp_{src}"]["total"] += int(df[src].notna().sum())
            unseen_log[f"otp_{src}"]["unseen"] += int(unseen.sum())
            df[dst] = mapped.fillna(global_otp).astype("float32")
    return df


def apply_track_b_feature_engineering(df):
    if "DEP_DELAY" in df.columns:
        df["IS_HEAVY_DELAY"] = (df["DEP_DELAY"] > 30).astype("Int16")
    return df


def build_ml_rows(df, target_col, freq_maps, otp_maps, global_otp, unseen_log):
    df = apply_freq_otp(df.copy(), freq_maps, otp_maps, global_otp, unseen_log)
    df = apply_track_b_feature_engineering(df)
    df = df[df[target_col].notna()].copy()
    if df.empty:
        return None, None
    a_cols = [c for c in TRACK_A_FEATURES if c in df.columns] + [target_col]
    track_a = df[a_cols].copy()
    b_cols = [c for c in TRACK_A_FEATURES + TRACK_B_EXTRA + TRACK_B_ENGINEERED if c in df.columns] + [target_col]
    track_b = df[b_cols].copy()
    return track_a, track_b


In [18]:
# Core pipeline and summary functions copied from pipeline.py and reporting.py

def load_mappings(mappings_dir):
    freq_maps = {}
    for col in FREQ_ENCODE_COLS:
        p = mappings_dir / f"freq_{col}.parquet"
        if p.exists():
            tmp = pd.read_parquet(p)
            freq_maps[col] = dict(zip(tmp["key"], tmp["freq"]))
    otp_maps = {}
    for col in OTP_GROUP_COLS:
        p = mappings_dir / f"otp_{col}.parquet"
        if p.exists():
            tmp = pd.read_parquet(p)
            otp_maps[col] = dict(zip(tmp["key"], tmp["otp"]))
    gp = mappings_dir / "global_otp.json"
    global_otp = json.loads(gp.read_text())["global_otp"] if gp.exists() else 0.5
    return freq_maps, otp_maps, global_otp


def run_pass1(input_files, chunksize, target_col, out_dir):
    mappings_dir = Path(out_dir) / "mappings"
    mappings_dir.mkdir(parents=True, exist_ok=True)
    meta_path = mappings_dir / "_meta.json"
    inp_hash = hash_inputs(input_files)
    if meta_path.exists():
        meta = json.loads(meta_path.read_text())
        if meta.get("completed_pass1") and meta.get("inputs_hash") == inp_hash and meta.get("chunksize") == chunksize and meta.get("target") == target_col:
            log.info("Pass 1 checkpoint found and valid; skipping.")
            return load_mappings(mappings_dir)
        log.info("Pass 1 checkpoint stale; re-running.")
    log.info("=== PASS 1: Building train-only mappings (years 2021-2024) ===")
    freq_counts = {c: {} for c in FREQ_ENCODE_COLS}
    otp_counts = {c: {} for c in OTP_GROUP_COLS}
    total_train_operated = 0
    train_files = [f for f in input_files if any(str(y) in str(f) for y in TRAIN_YEARS)]
    for fpath in train_files:
        log.info(f"  Pass1 reading {fpath}")
        for ci, chunk in enumerate(pd.read_csv(fpath, chunksize=chunksize, low_memory=False, usecols=lambda c: c.strip().upper() not in set(COLUMNS_TO_DROP))):
            chunk = standardize_columns(chunk)
            chunk = cast_dtypes(chunk)

            chunk = apply_r0_duplicates(chunk, None)
            chunk, _ = apply_r1_cancelled(chunk)
            chunk, _ = apply_r2_diverted(chunk)
            apply_r3_hhmm(chunk, {})
            apply_r4_date(chunk, {})
            apply_r6_route(chunk)
            operated = chunk[(chunk["CANCELLED"] == 0) & (chunk["DIVERTED"] == 0)]
            total_train_operated += len(operated)
            for col in FREQ_ENCODE_COLS:
                if col in chunk.columns:
                    vc = chunk[col].value_counts()
                    for k, v in vc.items():
                        freq_counts[col][k] = freq_counts[col].get(k, 0) + int(v)
            for col in OTP_GROUP_COLS:
                if col in operated.columns and target_col in operated.columns:
                    grp = operated.groupby(col)[target_col].agg(["count", "sum"])
                    for k, row in grp.iterrows():
                        prev = otp_counts[col].get(k, [0, 0])
                        otp_counts[col][k] = [prev[0] + int(row["count"]), prev[1] + int(row["sum"])]
            if (ci + 1) % 3 == 0:
                log.info(f"    chunk {ci+1} done")
    freq_maps = {}
    for col, counts in freq_counts.items():
        total = sum(counts.values()) or 1
        freq_maps[col] = {k: v / total for k, v in counts.items()}
    otp_maps = {}
    global_total = 0
    global_ontime_sum = 0
    for col, mapping in otp_counts.items():
        otp_maps[col] = {}
        for k, (cnt, delayed) in mapping.items():
            otp_maps[col][k] = (cnt - delayed) / cnt if cnt > 0 else 0.5
            global_total += cnt
            global_ontime_sum += (cnt - delayed)
    global_otp = global_ontime_sum / global_total if global_total > 0 else 0.5
    global_otp = ((total_train_operated - sum(v[1] for v in otp_counts["ORIGIN"].values())) / total_train_operated if total_train_operated > 0 else 0.5)
    for col, m in freq_maps.items():
        pd.DataFrame(list(m.items()), columns=["key", "freq"]).to_parquet(mappings_dir / f"freq_{col}.parquet", index=False)
    for col, m in otp_maps.items():
        pd.DataFrame(list(m.items()), columns=["key", "otp"]).to_parquet(mappings_dir / f"otp_{col}.parquet", index=False)
    (mappings_dir / "global_otp.json").write_text(json.dumps({"global_otp": global_otp}))
    meta = {"completed_pass1": True, "inputs_hash": inp_hash, "chunksize": chunksize, "target": target_col, "total_train_operated": total_train_operated}
    meta_path.write_text(json.dumps(meta, indent=2))
    log.info(f"Pass 1 complete. Train operated rows: {total_train_operated:,}")
    return freq_maps, otp_maps, global_otp


def write_parquet_partition(df, base_dir, year_val, writers_state):
    part_dir = Path(base_dir) / f"YEAR={year_val}"
    part_dir.mkdir(parents=True, exist_ok=True)
    fpath = part_dir / "part-0.parquet"
    df_write = df.drop(columns=["YEAR"], errors="ignore")
    table = pa.Table.from_pandas(df_write, preserve_index=False)
    key = str(fpath)
    if key not in writers_state:
        writers_state[key] = pq.ParquetWriter(str(fpath), table.schema, compression="snappy")
    try:
        writers_state[key].write_table(table)
    except (pa.ArrowInvalid, pa.ArrowTypeError):
        writers_state[key].close()
        writers_state[key] = pq.ParquetWriter(str(fpath), table.schema, compression="snappy")
        writers_state[key].write_table(table)


def run_pass2(input_files, chunksize, target_col, out_dir, overwrite, freq_maps, otp_maps, global_otp):
    log.info("=== PASS 2: Full cleaning and output generation ===")
    clean_full_dir = Path(out_dir) / "clean_full"
    clean_op_dir = Path(out_dir) / "clean_operated"
    ml_a_dir = Path(out_dir) / "ml_track_a"
    ml_b_dir = Path(out_dir) / "ml_track_b"
    for d in [clean_full_dir, clean_op_dir, ml_a_dir, ml_b_dir]:
        d.mkdir(parents=True, exist_ok=True)
    if overwrite:
        for d in [clean_full_dir, clean_op_dir]:
            for sub in d.glob("YEAR=*"):
                shutil.rmtree(sub, ignore_errors=True)
        for d in [ml_a_dir, ml_b_dir]:
            for p in d.glob("*.parquet"):
                p.unlink(missing_ok=True)
    hhmm_stats = {}
    dow_mismatches = {"count": 0}
    rule_counts = {}

    year_stats = {}
    writers_full = {}
    writers_op = {}
    unseen_log = {}
    ml_a_train, ml_a_test, ml_b_train, ml_b_test = [], [], [], []
    consistency = {"distance_neg": 0, "time_min_oor": 0}
    for fpath in input_files:
        log.info(f"  Pass2 reading {fpath}")
        for ci, chunk in enumerate(pd.read_csv(fpath, chunksize=chunksize, low_memory=False, usecols=lambda c: c.strip().upper() not in set(COLUMNS_TO_DROP))):
            chunk = standardize_columns(chunk)
            chunk = cast_dtypes(chunk)
            if "YEAR" not in chunk.columns or chunk["YEAR"].isna().all():
                log.warning(f"  chunk {ci} has no YEAR column, skipping")
                continue
            years_in_chunk = chunk["YEAR"].dropna().unique()
            chunk = clean_chunk(chunk, target_col, hhmm_stats, dow_mismatches, rule_counts)
            if "DISTANCE" in chunk.columns:
                consistency["distance_neg"] += int((chunk["DISTANCE"] < 0).sum())
            for yr in years_in_chunk:
                yr = int(yr)
                yr_chunk = chunk[chunk["YEAR"] == yr]
                st = year_stats.setdefault(yr, {"rows_read": 0, "rows_full": 0, "rows_operated": 0, "cancelled": 0, "diverted": 0})
                st["rows_read"] += len(yr_chunk)
                st["rows_full"] += len(yr_chunk)
                st["cancelled"] += int((yr_chunk["CANCELLED"] == 1).sum()) if "CANCELLED" in yr_chunk.columns else 0
                st["diverted"] += int((yr_chunk["DIVERTED"] == 1).sum()) if "DIVERTED" in yr_chunk.columns else 0
                write_parquet_partition(yr_chunk, clean_full_dir, yr, writers_full)
                op = yr_chunk[(yr_chunk["CANCELLED"] == 0) & (yr_chunk["DIVERTED"] == 0)]
                st["rows_operated"] += len(op)
                if not op.empty:
                    write_parquet_partition(op, clean_op_dir, yr, writers_op)
                    ta, tb = build_ml_rows(op, target_col, freq_maps, otp_maps, global_otp, unseen_log)
                    if ta is not None:
                        if yr in TRAIN_YEARS:
                            ml_a_train.append(ta)
                            ml_b_train.append(tb)
                        else:
                            ml_a_test.append(ta)
                            ml_b_test.append(tb)
            if (ci + 1) % 3 == 0:
                log.info(f"    chunk {ci+1} done")
    for w in list(writers_full.values()) + list(writers_op.values()):
        w.close()
    ml_counts = {}
    ml_missingness = {}
    for name, parts, d in [("ml_track_a_train", ml_a_train, ml_a_dir), ("ml_track_a_test", ml_a_test, ml_a_dir), ("ml_track_b_train", ml_b_train, ml_b_dir), ("ml_track_b_test", ml_b_test, ml_b_dir)]:
        if parts:
            combined = pd.concat(parts, ignore_index=True)
            ml_missingness[name] = [{"column": col, "dtype": str(combined[col].dtype), "missing_pct": round(float(combined[col].isna().mean() * 100), 2)} for col in combined.columns]
            out_path = d / f"{name}.parquet"
            out_path.unlink(missing_ok=True)
            combined.to_parquet(out_path, engine="pyarrow", compression="snappy", index=False)
            ml_counts[name] = len(combined)
            log.info(f"  Wrote {name}: {len(combined):,} rows, cols={list(combined.columns)}")
        else:
            ml_counts[name] = 0
            ml_missingness[name] = []
            log.warning(f"  {name} has 0 rows!")
    log.info("Pass 2 complete.")
    return {"year_stats": year_stats, "hhmm_stats": hhmm_stats, "dow_mismatches": dow_mismatches, "rule_counts": rule_counts, "consistency": consistency, "ml_counts": ml_counts, "ml_missingness": ml_missingness, "unseen_log": unseen_log}


In [19]:
def summarize_missingness(df):
    return [{"column": col, "dtype": str(df[col].dtype), "missing_pct": round(float(df[col].isna().mean() * 100), 2)} for col in df.columns]


def sample_schema_summary(out_dir):
    sample_dir = Path(out_dir) / "clean_full"
    parts = sorted(sample_dir.rglob("*.parquet"))
    if not parts:
        return {"sample_partition": None, "columns": [], "derived_columns": [], "error": None}
    try:
        sample_df = pd.read_parquet(parts[0], engine="pyarrow")
        rows = summarize_missingness(sample_df)
        derived = [c for c in sample_df.columns if c in DERIVED_COLUMNS]
        return {"sample_partition": str(parts[0]), "columns": rows, "derived_columns": derived, "error": None}
    except Exception as exc:
        return {"sample_partition": None, "columns": [], "derived_columns": [], "error": str(exc)}


def build_processing_context(input_files, chunksize, target_col, out_dir, stats, start_time, end_time):
    ys = stats["year_stats"]
    elapsed = (datetime.fromisoformat(end_time) - datetime.fromisoformat(start_time)).total_seconds()
    artifacts = [
        {"directory": f"{out_dir}/clean_full/", "description": "Full cleaned data", "partition": "YEAR=YYYY/part-0.parquet"},
        {"directory": f"{out_dir}/clean_operated/", "description": "Operated only", "partition": "YEAR=YYYY/part-0.parquet"},
        {"directory": f"{out_dir}/ml_track_a/", "description": "ML pre-flight", "partition": "ml_track_a_train.parquet, ml_track_a_test.parquet"},
        {"directory": f"{out_dir}/ml_track_b/", "description": "ML post-pushback", "partition": "ml_track_b_train.parquet, ml_track_b_test.parquet"},
        {"directory": f"{out_dir}/mappings/", "description": "Train-only freq/OTP maps", "partition": "Parquet files + global_otp.json"},
    ]
    year_rows = []
    for yr in sorted(ys.keys()):
        s = ys[yr]
        rows_read = s["rows_read"]
        year_rows.append({"YEAR": int(yr), "rows_read": int(rows_read), "rows_full": int(s["rows_full"]), "rows_operated": int(s["rows_operated"]), "cancelled": int(s["cancelled"]), "diverted": int(s["diverted"]), "cancelled_pct": round((s["cancelled"] / rows_read * 100) if rows_read else 0.0, 2), "diverted_pct": round((s["diverted"] / rows_read * 100) if rows_read else 0.0, 2)})
    return {"generated_at": datetime.now().isoformat(), "run_config": {"machine": "local (~16 GB RAM)", "python": sys.version.split()[0], "pandas": pd.__version__, "pyarrow": pa.__version__, "files": [str(f) for f in input_files], "chunksize": int(chunksize), "target": target_col, "start": start_time, "end": end_time, "runtime_seconds": round(elapsed, 1)}, "ingestion_summary": {"years": year_rows, "totals": {"rows_read": int(sum(s["rows_read"] for s in ys.values())), "rows_full": int(sum(s["rows_full"] for s in ys.values())), "rows_operated": int(sum(s["rows_operated"] for s in ys.values()))}}, "schema_summary": {"clean_full": sample_schema_summary(out_dir), "ml_missingness": stats["ml_missingness"]}, "transformation_log": {"rule_counts": {k: int(v) for k, v in stats["rule_counts"].items()}, "hhmm_stats": {k: {"parsed": int(v["parsed"]), "na": int(v["na"])} for k, v in stats["hhmm_stats"].items()}, "rules": {"R0": "Duplicate handling", "R1": "Cancelled nullification", "R2": "Diverted nullification", "R3": "HHMM parsing", "R4": "FL_DATE / DOW validation", "R5": "IS_WEEKEND", "R6": "ROUTE", "R9": "Operated subset", "R10": "Freq/OTP encoding", "R11": "Track B feature engineering"}}, "ml_audit": {"row_counts": {k: int(v) for k, v in stats["ml_counts"].items()}, "track_a_features": TRACK_A_FEATURES, "track_b_features": TRACK_A_FEATURES + TRACK_B_EXTRA + TRACK_B_ENGINEERED, "track_a_forbidden": sorted(TRACK_A_FORBIDDEN), "track_b_forbidden": sorted(TRACK_B_FORBIDDEN), "unseen_rates": {k: {"total": int(v["total"]), "unseen": int(v["unseen"]), "rate_pct": round((v["unseen"] / v["total"] * 100) if v["total"] else 0.0, 3)} for k, v in stats["unseen_log"].items()}, "unseen_policy": "Unseen values mapped to 0 (freq) or global OTP mean (OTP)."}, "artifacts": artifacts, "out_dir": str(out_dir)}


def acceptance_checks(out_dir, target_col):
    errors = []
    od = Path(out_dir)
    for yr in range(2021, 2026):
        for sub in ["clean_full", "clean_operated"]:
            d = od / sub / f"YEAR={yr}"
            if not d.exists():
                errors.append(f"Missing partition: {d}")
    for fname in ["ml_track_a/ml_track_a_train.parquet", "ml_track_a/ml_track_a_test.parquet", "ml_track_b/ml_track_b_train.parquet", "ml_track_b/ml_track_b_test.parquet"]:
        p = od / fname
        if not p.exists():
            errors.append(f"Missing ML file: {p}")
    if errors:
        raise AssertionError("ACCEPTANCE FAILED:\n" + "\n".join(f"  - {e}" for e in errors))
    print("Acceptance checks passed")


In [20]:
# Parameters
INPUT_FILES = FILES.copy()
CHUNK_SIZE = CHUNKSIZE
TARGET_COL = TARGET
OUT_DIR = REPO_ROOT / "data" / "processed"
OVERWRITE = True
summary = None
stats = None
params_df = pd.DataFrame([{"param": "INPUT_FILES", "value": INPUT_FILES}, {"param": "CHUNK_SIZE", "value": CHUNK_SIZE}, {"param": "TARGET_COL", "value": TARGET_COL}, {"param": "OUT_DIR", "value": str(OUT_DIR)}, {"param": "OVERWRITE", "value": OVERWRITE}])
display(params_df)


,param,value
0,INPUT_FILES,"[data/raw/T_ONTIME_REPORTING_2021.csv, data/ra..."
1,CHUNK_SIZE,700000
2,TARGET_COL,ARR_DEL15
3,OUT_DIR,C:\Users\Admin\Documents\GitHub\Airline_OTP_An...
4,OVERWRITE,True


## 1. Validate Inputs


In [21]:
required = {"YEAR", "DAY_OF_MONTH", "DAY_OF_WEEK", "OP_CARRIER", "ORIGIN", "DEST", "CRS_DEP_TIME", "CRS_ARR_TIME", "CANCELLED", "DIVERTED", TARGET_COL}
resolved_inputs = [REPO_ROOT / Path(p) for p in INPUT_FILES]
missing_files = [str(p) for p in resolved_inputs if not p.exists()]
if missing_files:
    raise FileNotFoundError(f"Missing input files: {missing_files}")
sample = pd.read_csv(resolved_inputs[0], nrows=0)
sample.columns = sample.columns.str.strip().str.upper()
missing_cols = sorted(required - set(sample.columns))
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")
print("Validated files:", len(resolved_inputs))
print("First file      :", resolved_inputs[0])
print("Target column   :", TARGET_COL)


Validated files: 5
First file      : C:\Users\Admin\Documents\GitHub\Airline_OTP_Analysis\data\raw\T_ONTIME_REPORTING_2021.csv
Target column   : ARR_DEL15


## 2. Run Preprocessing


In [22]:
start = datetime.now().isoformat()
freq_maps, otp_maps, global_otp = run_pass1(resolved_inputs, CHUNK_SIZE, TARGET_COL, OUT_DIR)
stats = run_pass2(resolved_inputs, CHUNK_SIZE, TARGET_COL, OUT_DIR, OVERWRITE, freq_maps, otp_maps, global_otp)
end = datetime.now().isoformat()
summary = build_processing_context(resolved_inputs, CHUNK_SIZE, TARGET_COL, OUT_DIR, stats, start, end)
print("Preprocessing complete")
print("Total rows processed:", f"{sum(s['rows_read'] for s in stats['year_stats'].values()):,}")
print("Runtime seconds     :", summary["run_config"]["runtime_seconds"])


2026-04-05 15:32:36,407 | INFO | Pass 1 checkpoint found and valid; skipping.
2026-04-05 15:32:36,443 | INFO | === PASS 2: Full cleaning and output generation ===
2026-04-05 15:32:36,483 | INFO |   Pass2 reading C:\Users\Admin\Documents\GitHub\Airline_OTP_Analysis\data\raw\T_ONTIME_REPORTING_2021.csv
2026-04-05 15:32:49,224 | INFO |   Pass2 reading C:\Users\Admin\Documents\GitHub\Airline_OTP_Analysis\data\raw\T_ONTIME_REPORTING_2022.csv
2026-04-05 15:33:09,290 | INFO |   Pass2 reading C:\Users\Admin\Documents\GitHub\Airline_OTP_Analysis\data\raw\T_ONTIME_REPORTING_2023.csv
2026-04-05 15:33:24,393 | INFO |   Pass2 reading C:\Users\Admin\Documents\GitHub\Airline_OTP_Analysis\data\raw\T_ONTIME_REPORTING_2024.csv
2026-04-05 15:33:41,292 | INFO |   Pass2 reading C:\Users\Admin\Documents\GitHub\Airline_OTP_Analysis\data\raw\T_ONTIME_REPORTING_2025.csv
2026-04-05 15:34:01,732 | INFO |   Wrote ml_track_a_train: 1,913,300 rows, cols=['YEAR', 'DAY_OF_MONTH', 'DAY_OF_WEEK', 'IS_WEEKEND', 'CRS_DEP

Preprocessing complete
Total rows processed: 2,525,183
Runtime seconds     : 89.6


## 3. Ingestion Summary


In [23]:
if summary is None:
    raise RuntimeError('Run preprocessing first.')
ingestion_df = pd.DataFrame(summary['ingestion_summary']['years'])
display(ingestion_df)
totals_df = pd.DataFrame([summary['ingestion_summary']['totals']])
display(totals_df)


,YEAR,rows_read,rows_full,rows_operated,cancelled,diverted,cancelled_pct,diverted_pct
0,2021,361428,361428,357204,3647,577,1.01,0.16
1,2022,537901,537901,503529,33255,1117,6.18,0.21
2,2023,538837,538837,527197,10295,1345,1.91,0.25
3,2024,547271,547271,525370,20389,1512,3.73,0.28
4,2025,539746,539746,522269,16311,1166,3.02,0.22


,rows_read,rows_full,rows_operated
0,2525183,2525183,2435569


## 4. Schema and Missingness


In [24]:
if summary is None:
    raise RuntimeError('Run preprocessing first.')
schema_summary = summary['schema_summary']
clean_full_summary = schema_summary['clean_full']
if clean_full_summary['error']:
    print('Schema read error:', clean_full_summary['error'])
else:
    print('Sample partition:', clean_full_summary['sample_partition'])
    display(pd.DataFrame(clean_full_summary['columns']))
    print('Derived columns:', clean_full_summary['derived_columns'])
for name, rows in schema_summary['ml_missingness'].items():
    print(f'ML missingness before write: {name}')
    if rows:
        display(pd.DataFrame(rows))
    else:
        print('No rows to write.')


Sample partition: C:\Users\Admin\Documents\GitHub\Airline_OTP_Analysis\data\processed\clean_full\YEAR=2021\part-0.parquet


,column,dtype,missing_pct
0,DAY_OF_MONTH,Int16,0.00
1,DAY_OF_WEEK,Int16,0.00
2,OP_UNIQUE_CARRIER,string,0.00
3,OP_CARRIER,string,0.00
4,ORIGIN,string,0.00
5,ORIGIN_CITY_NAME,string,0.00
6,ORIGIN_STATE_ABR,string,0.00
7,ORIGIN_STATE_NM,string,0.00
8,DEST,string,0.00
9,DEST_CITY_NAME,string,0.00


Derived columns: ['CRS_DEP_TIME_MIN', 'CRS_ARR_TIME_MIN', 'CRS_DEP_SIN', 'CRS_DEP_COS', 'CRS_ARR_SIN', 'CRS_ARR_COS', 'FL_DATE', 'IS_WEEKEND', 'ROUTE']
ML missingness before write: ml_track_a_train


,column,dtype,missing_pct
0,YEAR,Int16,0.0
1,DAY_OF_MONTH,Int16,0.0
2,DAY_OF_WEEK,Int16,0.0
3,IS_WEEKEND,Int16,0.0
4,CRS_DEP_TIME_MIN,Int32,0.0
5,CRS_ARR_TIME_MIN,Int32,0.0
6,CRS_DEP_SIN,float32,0.0
7,CRS_DEP_COS,float32,0.0
8,CRS_ARR_SIN,float32,0.0
9,CRS_ARR_COS,float32,0.0


ML missingness before write: ml_track_a_test


,column,dtype,missing_pct
0,YEAR,Int16,0.0
1,DAY_OF_MONTH,Int16,0.0
2,DAY_OF_WEEK,Int16,0.0
3,IS_WEEKEND,Int16,0.0
4,CRS_DEP_TIME_MIN,Int32,0.0
5,CRS_ARR_TIME_MIN,Int32,0.0
6,CRS_DEP_SIN,float32,0.0
7,CRS_DEP_COS,float32,0.0
8,CRS_ARR_SIN,float32,0.0
9,CRS_ARR_COS,float32,0.0


ML missingness before write: ml_track_b_train


,column,dtype,missing_pct
0,YEAR,Int16,0.0
1,DAY_OF_MONTH,Int16,0.0
2,DAY_OF_WEEK,Int16,0.0
3,IS_WEEKEND,Int16,0.0
4,CRS_DEP_TIME_MIN,Int32,0.0
5,CRS_ARR_TIME_MIN,Int32,0.0
6,CRS_DEP_SIN,float32,0.0
7,CRS_DEP_COS,float32,0.0
8,CRS_ARR_SIN,float32,0.0
9,CRS_ARR_COS,float32,0.0


ML missingness before write: ml_track_b_test


,column,dtype,missing_pct
0,YEAR,Int16,0.0
1,DAY_OF_MONTH,Int16,0.0
2,DAY_OF_WEEK,Int16,0.0
3,IS_WEEKEND,Int16,0.0
4,CRS_DEP_TIME_MIN,Int32,0.0
5,CRS_ARR_TIME_MIN,Int32,0.0
6,CRS_DEP_SIN,float32,0.0
7,CRS_DEP_COS,float32,0.0
8,CRS_ARR_SIN,float32,0.0
9,CRS_ARR_COS,float32,0.0


## 5. Transformation Log


In [25]:
if summary is None:
    raise RuntimeError('Run preprocessing first.')
rule_counts = summary['transformation_log']['rule_counts']
rules = summary['transformation_log']['rules']
rule_rows = []
for key, desc in rules.items():
    rule_rows.append({'rule': key, 'description': desc, 'count': rule_counts.get(key)})
display(pd.DataFrame(rule_rows))
hhmm_df = pd.DataFrame([{'column': k, 'parsed': v['parsed'], 'na': v['na']} for k, v in summary['transformation_log']['hhmm_stats'].items()])
display(hhmm_df)


,rule,description,count
0,R0,Duplicate handling,2.0
1,R1,Cancelled nullification,83897.0
2,R2,Diverted nullification,5717.0
3,R3,HHMM parsing,2525183.0
4,R4,FL_DATE / DOW validation,2525183.0
5,R5,IS_WEEKEND,NaN
6,R6,ROUTE,NaN
7,R9,Operated subset,NaN
8,R10,Freq/OTP encoding,NaN
9,R11,Track B feature engineering,NaN


,column,parsed,na
0,CRS_DEP_TIME,2525183,0
1,CRS_ARR_TIME,2525183,0


## 7. ML Dataset Audit


In [26]:
if summary is None:
    raise RuntimeError('Run preprocessing first.')
ml_summary = summary['ml_audit']
display(pd.DataFrame(ml_summary['row_counts'].items(), columns=['dataset', 'rows']))
display(pd.DataFrame(ml_summary['unseen_rates']).T.reset_index().rename(columns={'index': 'source'}))
print('Track A features:', ml_summary['track_a_features'])
print('Track B features:', ml_summary['track_b_features'])
print('Track A forbidden:', ml_summary['track_a_forbidden'])
print('Track B forbidden:', ml_summary['track_b_forbidden'])
print(ml_summary['unseen_policy'])


,dataset,rows
0,ml_track_a_train,1913300
1,ml_track_a_test,522269
2,ml_track_b_train,1913300
3,ml_track_b_test,522269


,source,total,unseen,rate_pct
0,OP_CARRIER,2435569.0,0.0,0.000
1,ORIGIN,2435569.0,77.0,0.003
2,DEST,2435569.0,78.0,0.003
3,ROUTE,2435569.0,3541.0,0.145
4,DEP_TIME_BLK,2435569.0,0.0,0.000
5,otp_ORIGIN,2435569.0,77.0,0.003
6,otp_OP_CARRIER,2435569.0,0.0,0.000


Track A features: ['YEAR', 'DAY_OF_MONTH', 'DAY_OF_WEEK', 'IS_WEEKEND', 'CRS_DEP_TIME_MIN', 'CRS_ARR_TIME_MIN', 'CRS_DEP_SIN', 'CRS_DEP_COS', 'CRS_ARR_SIN', 'CRS_ARR_COS', 'DISTANCE', 'OP_CARRIER_FREQ', 'CARRIER_HIST_OTP', 'ORIGIN_FREQ', 'ORIGIN_HIST_OTP', 'DEST_FREQ', 'ROUTE_FREQ', 'DEP_TIME_BLK_FREQ']
Track B features: ['YEAR', 'DAY_OF_MONTH', 'DAY_OF_WEEK', 'IS_WEEKEND', 'CRS_DEP_TIME_MIN', 'CRS_ARR_TIME_MIN', 'CRS_DEP_SIN', 'CRS_DEP_COS', 'CRS_ARR_SIN', 'CRS_ARR_COS', 'DISTANCE', 'OP_CARRIER_FREQ', 'CARRIER_HIST_OTP', 'ORIGIN_FREQ', 'ORIGIN_HIST_OTP', 'DEST_FREQ', 'ROUTE_FREQ', 'DEP_TIME_BLK_FREQ', 'DEP_DELAY', 'TAXI_OUT', 'IS_HEAVY_DELAY']
Track A forbidden: ['ACTUAL_ELAPSED_TIME', 'AIR_TIME', 'ARR_DEL15', 'ARR_DELAY', 'ARR_DELAY_GROUP', 'ARR_DELAY_NEW', 'ARR_TIME', 'CANCELLED', 'CARRIER_DELAY', 'DEP_DEL15', 'DEP_DELAY', 'DEP_DELAY_GROUP', 'DEP_DELAY_NEW', 'DEP_TIME', 'DIVERTED', 'FIRST_DEP_TIME', 'LATE_AIRCRAFT_DELAY', 'LONGEST_ADD_GTIME', 'NAS_DELAY', 'SECURITY_DELAY', 'TAXI_IN'

## 8. Artifacts Produced


In [27]:
if summary is None:
    raise RuntimeError('Run preprocessing first.')
display(pd.DataFrame(summary['artifacts']))


,directory,description,partition
0,C:\Users\Admin\Documents\GitHub\Airline_OTP_An...,Full cleaned data,YEAR=YYYY/part-0.parquet
1,C:\Users\Admin\Documents\GitHub\Airline_OTP_An...,Operated only,YEAR=YYYY/part-0.parquet
2,C:\Users\Admin\Documents\GitHub\Airline_OTP_An...,ML pre-flight,"ml_track_a_train.parquet, ml_track_a_test.parquet"
3,C:\Users\Admin\Documents\GitHub\Airline_OTP_An...,ML post-pushback,"ml_track_b_train.parquet, ml_track_b_test.parquet"
4,C:\Users\Admin\Documents\GitHub\Airline_OTP_An...,Train-only freq/OTP maps,Parquet files + global_otp.json


## 9. Optional Direct Reads


In [28]:
clean_paths = sorted((Path(OUT_DIR) / 'clean_full').rglob('*.parquet'))
if clean_paths:
    clean_sample_df = pd.read_parquet(clean_paths[0])
    print('Clean sample shape:', clean_sample_df.shape)
    display(clean_sample_df.head())
else:
    print('No clean_full parquet files found.')
track_a_train = Path(OUT_DIR) / 'ml_track_a' / 'ml_track_a_train.parquet'
if track_a_train.exists():
    track_a_df = pd.read_parquet(track_a_train)
    print('Track A train shape:', track_a_df.shape)
    display(track_a_df.head())
else:
    print('Track A train parquet not found.')


Clean sample shape: (361428, 55)


,DAY_OF_MONTH,DAY_OF_WEEK,OP_UNIQUE_CARRIER,OP_CARRIER,ORIGIN,ORIGIN_CITY_NAME,ORIGIN_STATE_ABR,ORIGIN_STATE_NM,DEST,DEST_CITY_NAME,...,DIV_DISTANCE,CRS_DEP_TIME_MIN,CRS_ARR_TIME_MIN,CRS_DEP_SIN,CRS_DEP_COS,CRS_ARR_SIN,CRS_ARR_COS,FL_DATE,IS_WEEKEND,ROUTE
0,1,4,9E,9E,ATL,"Atlanta, GA",GA,Georgia,JAN,"Jackson/Vicksburg, MS",...,NaN,630,653,0.382683,-9.238795e-01,0.288196,-0.957571,2021-01-01,0,ATL-JAN
1,1,4,9E,9E,JAN,"Jackson/Vicksburg, MS",MS,Mississippi,ATL,"Atlanta, GA",...,NaN,705,851,0.065403,-9.978589e-01,-0.540974,-0.841039,2021-01-01,0,JAN-ATL
2,1,4,9E,9E,ATL,"Atlanta, GA",GA,Georgia,GSP,"Greer, SC",...,NaN,1230,1281,-0.793353,6.087615e-01,-0.639439,0.768842,2021-01-01,0,ATL-GSP
3,1,4,9E,9E,OKC,"Oklahoma City, OK",OK,Oklahoma,ATL,"Atlanta, GA",...,NaN,360,543,1.000000,-4.371139e-08,0.697790,-0.716302,2021-01-01,0,OKC-ATL
4,1,4,9E,9E,BHM,"Birmingham, AL",AL,Alabama,ATL,"Atlanta, GA",...,NaN,905,1024,-0.722364,-6.915130e-01,-0.970296,-0.241922,2021-01-01,0,BHM-ATL


Track A train shape: (1913300, 19)


,YEAR,DAY_OF_MONTH,DAY_OF_WEEK,IS_WEEKEND,CRS_DEP_TIME_MIN,CRS_ARR_TIME_MIN,CRS_DEP_SIN,CRS_DEP_COS,CRS_ARR_SIN,CRS_ARR_COS,DISTANCE,OP_CARRIER_FREQ,CARRIER_HIST_OTP,ORIGIN_FREQ,ORIGIN_HIST_OTP,DEST_FREQ,ROUTE_FREQ,DEP_TIME_BLK_FREQ,ARR_DEL15
0,2021,1,4,0,630,653,0.382683,-9.238795e-01,0.288196,-0.957571,341.0,0.037797,0.834555,0.050714,0.827204,0.001014,0.000430,0.066428,1
1,2021,1,4,0,705,851,0.065403,-9.978589e-01,-0.540974,-0.841039,341.0,0.037797,0.834555,0.001012,0.874104,0.050692,0.000429,0.064720,1
2,2021,1,4,0,1230,1281,-0.793353,6.087615e-01,-0.639439,0.768842,153.0,0.037797,0.834555,0.050714,0.827204,0.002002,0.000503,0.042448,0
3,2021,1,4,0,360,543,1.000000,-4.371139e-08,0.697790,-0.716302,761.0,0.037797,0.834555,0.003126,0.827345,0.050692,0.000329,0.070455,1
4,2021,1,4,0,905,1024,-0.722364,-6.915130e-01,-0.970296,-0.241922,134.0,0.037797,0.834555,0.002196,0.827603,0.050692,0.000434,0.059889,0
